<a href="https://colab.research.google.com/github/Yasmina3/CBRA-FYP/blob/main/Data_alignement_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# ALIGN REAL + SIMULATED CBRA 40-SENSOR TIME-SERIES DATASETS
# ============================================================

import os
import re
import numpy as np
import pandas as pd

# -----------------------------
# 0) SET YOUR FOLDER PATH HERE
# -----------------------------
FOLDER_PATH = "/content/drive/MyDrive/CBRA_STATIC_MODELING/INPUT"  # <-- CHANGE THIS

REAL_FILENAME = "real_breast_level_40sensors_imputed_final.csv"
SIM_FILENAME  = "calibrated_passive_generated_timeseries_long.csv"

REAL_PATH = os.path.join(FOLDER_PATH, REAL_FILENAME)
SIM_PATH  = os.path.join(FOLDER_PATH, SIM_FILENAME)

TARGET_T = 585

OUT_REAL = os.path.join(FOLDER_PATH, "aligned_real_imputed_585_long.csv")
OUT_SIM  = os.path.join(FOLDER_PATH, "aligned_simulated_passive_585_long.csv")
OUT_COMBINED = os.path.join(FOLDER_PATH, "aligned_combined_real_sim_585_long.csv")
OUT_COMBINED_LABELED = os.path.join(FOLDER_PATH, "aligned_combined_labeled_only.csv")
OUT_META = os.path.join(FOLDER_PATH, "aligned_sample_metadata.csv")

print("Real path:", REAL_PATH)
print("Sim path :", SIM_PATH)

# -----------------------------
# 1) LOAD DATA
# -----------------------------
real_df = pd.read_csv(REAL_PATH)
sim_df = pd.read_csv(SIM_PATH)

print("\nLoaded:")
print("Real shape:", real_df.shape)
print("Sim shape :", sim_df.shape)

# -----------------------------
# 2) SENSOR COLUMNS
# -----------------------------
real_sensor_cols = [f"s_{i:02d}" for i in range(1, 41)]
sim_sensor_old = [f"s_{i}" for i in range(1, 41)]
sim_sensor_new = [f"s_{i:02d}" for i in range(1, 41)]

missing_real_sensors = [c for c in real_sensor_cols if c not in real_df.columns]
missing_sim_sensors = [c for c in sim_sensor_old if c not in sim_df.columns]

if missing_real_sensors:
    raise ValueError(f"Missing real sensor columns: {missing_real_sensors}")

if missing_sim_sensors:
    raise ValueError(f"Missing simulated sensor columns: {missing_sim_sensors}")

# Rename simulated sensors s_1...s_40 -> s_01...s_40
sim_df = sim_df.rename(columns={old: new for old, new in zip(sim_sensor_old, sim_sensor_new)})

sensor_cols = real_sensor_cols
mask_cols = [f"m_{i:02d}" for i in range(1, 41)]

# -----------------------------
# 3) HELPER FUNCTIONS
# -----------------------------
def to_numeric_safe(s):
    return pd.to_numeric(s, errors="coerce")

def clean_real_density(x):
    """
    Cleans real breast density into A/B/C/D/unknown.
    Avoids accidentally extracting letters from words like 'Non disponible'.
    """
    if pd.isna(x):
        return "unknown"
    txt = str(x).strip().lower()

    if txt in ["", "nan", "none", "non disponible", "non-disponible", "not available", "unknown"]:
        return "unknown"

    txt = txt.replace("type", "type ").replace("acr", "acr ")

    m = re.search(r"\b(type|acr)\s*([abcd])\b", txt)
    if m:
        return m.group(2).upper()

    m = re.search(r"\b([abcd])\b", txt)
    if m:
        return m.group(1).upper()

    return "unknown"

def clean_sim_density_from_gland_size(x):
    """
    Maps simulated gland size to A/B/C-like density.
    This is an alignment proxy, not a perfect clinical BI-RADS label.
    """
    if pd.isna(x):
        return "unknown"

    try:
        v = float(x)
    except Exception:
        return "unknown"

    if np.isclose(v, 10):
        return "A"
    if np.isclose(v, 20):
        return "B"
    if np.isclose(v, 25):
        return "C"
    return "unknown"

def map_sim_condition_to_label(x):
    if pd.isna(x):
        return np.nan
    txt = str(x).strip().lower()
    if txt == "healthy":
        return 0
    if txt == "tumor":
        return 1
    return np.nan

def safe_first(series, default=np.nan):
    if series is None:
        return default
    s = series.dropna()
    if len(s) == 0:
        return default
    return s.iloc[0]

def make_norm_time_from_time_step(df, time_col):
    t = to_numeric_safe(df[time_col]).values.astype(float)
    if len(t) == 1 or np.nanmax(t) == np.nanmin(t):
        return np.zeros(len(t), dtype=np.float32)
    return ((t - np.nanmin(t)) / (np.nanmax(t) - np.nanmin(t))).astype(np.float32)

def interpolate_series(old_t, values, new_t):
    """
    Interpolates one sensor over normalized time.
    If values contain NaNs, interpolate through valid points.
    """
    values = np.asarray(values, dtype=float)
    old_t = np.asarray(old_t, dtype=float)

    valid = np.isfinite(old_t) & np.isfinite(values)

    if valid.sum() == 0:
        return np.full(len(new_t), np.nan, dtype=np.float32)

    if valid.sum() == 1:
        return np.full(len(new_t), values[valid][0], dtype=np.float32)

    return np.interp(new_t, old_t[valid], values[valid]).astype(np.float32)

def add_common_derived_channels(df, sensor_cols):
    """
    Adds lightweight model-ready channels:
    - row_mean_temp
    - ambient-delta sensors: ad_01...ad_40
    - spatial-centered sensors: sc_01...sc_40
    - quadrant-deviation sensors: qd_01...qd_40

    These are useful because they reduce dependence on absolute temperature level.
    """
    df = df.copy()

    sensor_values = df[sensor_cols].astype("float32")
    df["row_mean_temp"] = sensor_values.mean(axis=1).astype("float32")
    df["row_std_temp"] = sensor_values.std(axis=1).astype("float32")
    df["row_min_temp"] = sensor_values.min(axis=1).astype("float32")
    df["row_max_temp"] = sensor_values.max(axis=1).astype("float32")
    df["row_range_temp"] = (df["row_max_temp"] - df["row_min_temp"]).astype("float32")

    # Ambient-normalized sensors
    if "ambient_temp_c" in df.columns:
        amb = pd.to_numeric(df["ambient_temp_c"], errors="coerce").astype("float32")
        for c in sensor_cols:
            idx = c.split("_")[1]
            df[f"ad_{idx}"] = (df[c].astype("float32") - amb).astype("float32")

    # Spatial-centered sensors
    for c in sensor_cols:
        idx = c.split("_")[1]
        df[f"sc_{idx}"] = (df[c].astype("float32") - df["row_mean_temp"]).astype("float32")

    # Quadrant means
    quadrants = {
        "SI": [f"s_{i:02d}" for i in range(1, 11)],
        "SE": [f"s_{i:02d}" for i in range(11, 21)],
        "IE": [f"s_{i:02d}" for i in range(21, 31)],
        "II": [f"s_{i:02d}" for i in range(31, 41)],
    }

    for q, cols in quadrants.items():
        df[f"q_{q}_mean"] = df[cols].astype("float32").mean(axis=1).astype("float32")

    # Quadrant deviation sensors
    for q, cols in quadrants.items():
        qmean = df[f"q_{q}_mean"].astype("float32")
        for c in cols:
            idx = c.split("_")[1]
            df[f"qd_{idx}"] = (df[c].astype("float32") - qmean).astype("float32")

    # Useful quadrant contrasts
    df["qdiff_IE_SI"] = (df["q_IE_mean"] - df["q_SI_mean"]).astype("float32")
    df["qdiff_II_SI"] = (df["q_II_mean"] - df["q_SI_mean"]).astype("float32")
    df["qdiff_SE_SI"] = (df["q_SE_mean"] - df["q_SI_mean"]).astype("float32")
    df["qdiff_IE_SE"] = (df["q_IE_mean"] - df["q_SE_mean"]).astype("float32")
    df["qdiff_II_SE"] = (df["q_II_mean"] - df["q_SE_mean"]).astype("float32")
    df["qdiff_II_IE"] = (df["q_II_mean"] - df["q_IE_mean"]).astype("float32")

    return df

def add_temporal_slopes(df, sensor_cols):
    """
    Adds first-difference temporal slope channels sl_01...sl_40 per sample.
    """
    df = df.sort_values(["domain", "sample_uid", "time_idx"]).copy()

    for c in sensor_cols:
        idx = c.split("_")[1]
        df[f"sl_{idx}"] = (
            df.groupby(["domain", "sample_uid"], sort=False)[c]
              .diff()
              .fillna(0)
              .astype("float32")
        )

    return df

# -----------------------------
# 4) BUILD REAL ALIGNED DATA
# -----------------------------
def build_real_aligned(real_df):
    df = real_df.copy()

    # Keep physiologically valid values only
    for c in sensor_cols:
        df[c] = to_numeric_safe(df[c])
        df.loc[(df[c] < 20) | (df[c] > 45), c] = np.nan

    # Required IDs
    if "sample_id" not in df.columns:
        raise ValueError("Real dataset must contain sample_id.")
    if "time_step" not in df.columns:
        raise ValueError("Real dataset must contain time_step.")

    if "test_ref" in df.columns:
        group_col = "test_ref"
    elif "group_id" in df.columns:
        group_col = "group_id"
    else:
        group_col = "sample_id"

    new_time = np.linspace(0, 1, TARGET_T, dtype=np.float32)
    all_samples = []

    sample_ids = sorted(df["sample_id"].dropna().unique())
    print(f"\nReal samples found: {len(sample_ids)}")

    for sid in sample_ids:
        g = df[df["sample_id"] == sid].copy()
        g = g.sort_values("time_step")

        # Collapse duplicate time steps if they exist
        # Sensor values averaged; metadata kept later from original group.
        g["_norm_time_original"] = make_norm_time_from_time_step(g, "time_step")

        old_t = g["_norm_time_original"].values.astype(float)

        out = pd.DataFrame({
            "domain": "real",
            "source_type": "clinical_real_imputed",
            "sample_uid": str(sid),
            "group_uid": str(safe_first(g[group_col])) if group_col in g.columns else str(sid),
            "time_idx": np.arange(TARGET_T, dtype=np.int32),
            "norm_time": new_time,
        })

        # Label
        if "breast_label" in g.columns:
            label_value = safe_first(g["breast_label"])
            out["label"] = label_value
        else:
            out["label"] = np.nan

        out["label"] = pd.to_numeric(out["label"], errors="coerce")
        out["supervised_label_available"] = out["label"].notna().astype(np.int8)

        # Side
        if "side" in g.columns:
            out["side"] = str(safe_first(g["side"], default="unknown"))
        else:
            out["side"] = "unknown"

        # Density
        if "meta_breast_density_raw" in g.columns:
            out["density_clean"] = clean_real_density(safe_first(g["meta_breast_density_raw"]))
        else:
            out["density_clean"] = "unknown"
        out["density_source"] = "real_meta_breast_density_raw"

        # Ambient temperature
        if "meta_room_temp" in g.columns:
            out["ambient_temp_c"] = pd.to_numeric(safe_first(g["meta_room_temp"]), errors="coerce")
        else:
            out["ambient_temp_c"] = np.nan

        # Patient/core temperature
        if "meta_patient_temp" in g.columns:
            out["patient_or_core_temp_c"] = pd.to_numeric(safe_first(g["meta_patient_temp"]), errors="coerce")
        else:
            out["patient_or_core_temp_c"] = np.nan

        # Row-level imputation flag, resampled approximately
        if "any_sensor_imputed" in g.columns:
            imp_raw = pd.to_numeric(g["any_sensor_imputed"], errors="coerce").fillna(0).values.astype(float)
            imp_interp = interpolate_series(old_t, imp_raw, new_time)
            out["imputation_flag"] = (imp_interp > 0.5).astype(np.int8)
        else:
            out["imputation_flag"] = 0

        # Real-only physical sim fields are unknown
        out["gland_size_mm"] = np.nan
        out["ambient_temp_source"] = "real_meta_room_temp"
        out["tumor_radius_mm"] = np.nan
        out["tumor_depth_mm"] = np.nan
        out["tumor_heat"] = np.nan
        out["fat_heat"] = np.nan
        out["gland_heat"] = np.nan
        out["tumor_x"] = np.nan
        out["tumor_y"] = np.nan
        out["tumor_z"] = np.nan

        # Interpolate sensors
        for c in sensor_cols:
            out[c] = interpolate_series(old_t, g[c].values, new_time)

        # Mask handling:
        # Since this is already imputed data, exact per-sensor original masks may not exist.
        # If exact mask columns exist, use them. Else set m_i=1 and preserve imputation_flag.
        for c, m in zip(sensor_cols, mask_cols):
            exact_candidates = [
                f"{c}_observed",
                f"{c}_mask",
                f"m_{c.split('_')[1]}",
                f"was_observed_{c}",
            ]
            found = None
            for cand in exact_candidates:
                if cand in g.columns:
                    found = cand
                    break

            if found is not None:
                raw_mask = pd.to_numeric(g[found], errors="coerce").fillna(1).values.astype(float)
                out[m] = (interpolate_series(old_t, raw_mask, new_time) > 0.5).astype(np.int8)
            else:
                out[m] = 1

        all_samples.append(out)

    aligned = pd.concat(all_samples, ignore_index=True)

    # Optimize types
    for c in sensor_cols:
        aligned[c] = aligned[c].astype("float32")
    for c in mask_cols:
        aligned[c] = aligned[c].astype("int8")

    aligned["label"] = pd.to_numeric(aligned["label"], errors="coerce")
    aligned["time_idx"] = aligned["time_idx"].astype("int32")
    aligned["norm_time"] = aligned["norm_time"].astype("float32")

    return aligned

# -----------------------------
# 5) BUILD SIMULATED ALIGNED DATA
# -----------------------------
def build_sim_aligned(sim_df):
    df = sim_df.copy()

    # Clean sensors
    for c in sensor_cols:
        df[c] = to_numeric_safe(df[c])
        df.loc[(df[c] < 20) | (df[c] > 45), c] = np.nan

    if "generated_id" in df.columns:
        sample_col = "generated_id"
    elif "sample_id" in df.columns:
        sample_col = "sample_id"
    else:
        raise ValueError("Simulated dataset must contain generated_id or sample_id.")

    if "time_step" not in df.columns:
        raise ValueError("Simulated dataset must contain time_step.")

    # Label
    if "condition" in df.columns:
        df["_label"] = df["condition"].apply(map_sim_condition_to_label)
    else:
        df["_label"] = np.nan

    # Grouping
    if "source_static_id" in df.columns:
        group_col = "source_static_id"
    elif "parameter_signature" in df.columns:
        group_col = "parameter_signature"
    else:
        group_col = sample_col

    new_time = np.linspace(0, 1, TARGET_T, dtype=np.float32)
    all_samples = []

    sample_ids = sorted(df[sample_col].dropna().unique())
    print(f"Simulated samples found: {len(sample_ids)}")

    for sid in sample_ids:
        g = df[df[sample_col] == sid].copy()
        g = g.sort_values("time_step")

        if "norm_time" in g.columns:
            old_t = pd.to_numeric(g["norm_time"], errors="coerce").values.astype(float)
            if not np.isfinite(old_t).any() or np.nanmax(old_t) == np.nanmin(old_t):
                old_t = make_norm_time_from_time_step(g, "time_step")
        else:
            old_t = make_norm_time_from_time_step(g, "time_step")

        out = pd.DataFrame({
            "domain": "sim",
            "source_type": "calibrated_passive_simulated",
            "sample_uid": str(sid),
            "group_uid": str(safe_first(g[group_col])) if group_col in g.columns else str(sid),
            "time_idx": np.arange(TARGET_T, dtype=np.int32),
            "norm_time": new_time,
        })

        out["label"] = safe_first(g["_label"])
        out["label"] = pd.to_numeric(out["label"], errors="coerce")
        out["supervised_label_available"] = out["label"].notna().astype(np.int8)

        out["side"] = "simulated"

        # Density alignment
        if "gland_size_mm" in g.columns:
            gland_size = safe_first(g["gland_size_mm"])
            out["gland_size_mm"] = pd.to_numeric(gland_size, errors="coerce")
            out["density_clean"] = clean_sim_density_from_gland_size(gland_size)
        else:
            out["gland_size_mm"] = np.nan
            out["density_clean"] = "unknown"

        out["density_source"] = "sim_gland_size_mm"

        # Ambient
        if "ambient_temp_c" in g.columns:
            out["ambient_temp_c"] = pd.to_numeric(safe_first(g["ambient_temp_c"]), errors="coerce")
        else:
            out["ambient_temp_c"] = np.nan

        out["ambient_temp_source"] = "sim_ambient_temp_c"

        # Simulator assumes core/arterial reference around 37 C
        out["patient_or_core_temp_c"] = 37.0

        # No real imputation unless artificial domain randomization is added later
        out["imputation_flag"] = 0

        # Sim-only physical fields
        if "radius_mm" in g.columns:
            out["tumor_radius_mm"] = pd.to_numeric(safe_first(g["radius_mm"]), errors="coerce")
        elif "t_r" in g.columns:
            out["tumor_radius_mm"] = pd.to_numeric(safe_first(g["t_r"]), errors="coerce")
        else:
            out["tumor_radius_mm"] = np.nan

        if "d_tumor_mm" in g.columns:
            out["tumor_depth_mm"] = pd.to_numeric(safe_first(g["d_tumor_mm"]), errors="coerce")
        else:
            out["tumor_depth_mm"] = np.nan

        for sim_col, common_col in [
            ("tum_heat", "tumor_heat"),
            ("fat_heat", "fat_heat"),
            ("gland_heat", "gland_heat"),
            ("tumor_x", "tumor_x"),
            ("tumor_y", "tumor_y"),
            ("tumor_z", "tumor_z"),
        ]:
            if sim_col in g.columns:
                out[common_col] = pd.to_numeric(safe_first(g[sim_col]), errors="coerce")
            else:
                out[common_col] = np.nan

        # Interpolate/resample sensors
        for c in sensor_cols:
            out[c] = interpolate_series(old_t, g[c].values, new_time)

        # Simulated data are fully observed at this stage
        for m in mask_cols:
            out[m] = 1

        all_samples.append(out)

    aligned = pd.concat(all_samples, ignore_index=True)

    for c in sensor_cols:
        aligned[c] = aligned[c].astype("float32")
    for c in mask_cols:
        aligned[c] = aligned[c].astype("int8")

    aligned["label"] = pd.to_numeric(aligned["label"], errors="coerce")
    aligned["time_idx"] = aligned["time_idx"].astype("int32")
    aligned["norm_time"] = aligned["norm_time"].astype("float32")

    return aligned

# -----------------------------
# 6) ALIGN DATASETS
# -----------------------------
aligned_real = build_real_aligned(real_df)
aligned_sim = build_sim_aligned(sim_df)

print("\nAligned shapes before derived channels:")
print("Aligned real:", aligned_real.shape)
print("Aligned sim :", aligned_sim.shape)

# -----------------------------
# 7) ADD COMMON DERIVED CHANNELS
# -----------------------------
print("\nAdding derived common channels...")
aligned_real = add_common_derived_channels(aligned_real, sensor_cols)
aligned_sim = add_common_derived_channels(aligned_sim, sensor_cols)

aligned_real = add_temporal_slopes(aligned_real, sensor_cols)
aligned_sim = add_temporal_slopes(aligned_sim, sensor_cols)

# Ensure same columns in both
all_cols = sorted(set(aligned_real.columns).union(set(aligned_sim.columns)))

for c in all_cols:
    if c not in aligned_real.columns:
        aligned_real[c] = np.nan
    if c not in aligned_sim.columns:
        aligned_sim[c] = np.nan

aligned_real = aligned_real[all_cols]
aligned_sim = aligned_sim[all_cols]

combined = pd.concat([aligned_real, aligned_sim], ignore_index=True)

# Put important columns first
front_cols = [
    "domain", "source_type", "sample_uid", "group_uid",
    "label", "supervised_label_available",
    "side", "time_idx", "norm_time",
    "density_clean", "density_source",
    "ambient_temp_c", "ambient_temp_source",
    "patient_or_core_temp_c", "imputation_flag",
    "gland_size_mm", "tumor_radius_mm", "tumor_depth_mm",
    "tumor_heat", "fat_heat", "gland_heat",
    "tumor_x", "tumor_y", "tumor_z",
]
front_cols = [c for c in front_cols if c in combined.columns]
remaining_cols = [c for c in combined.columns if c not in front_cols]
combined = combined[front_cols + remaining_cols]

aligned_real = combined[combined["domain"] == "real"].copy()
aligned_sim = combined[combined["domain"] == "sim"].copy()

combined_labeled = combined[combined["supervised_label_available"] == 1].copy()

# -----------------------------
# 8) SAMPLE-LEVEL METADATA
# -----------------------------
meta_cols = [
    "domain", "source_type", "sample_uid", "group_uid",
    "label", "supervised_label_available", "side",
    "density_clean", "density_source",
    "ambient_temp_c", "ambient_temp_source",
    "patient_or_core_temp_c", "imputation_flag",
    "gland_size_mm", "tumor_radius_mm", "tumor_depth_mm",
    "tumor_heat", "fat_heat", "gland_heat",
    "tumor_x", "tumor_y", "tumor_z",
]

meta_cols = [c for c in meta_cols if c in combined.columns]

sample_metadata = (
    combined[meta_cols]
    .groupby(["domain", "sample_uid"], as_index=False)
    .agg({
        **{c: "first" for c in meta_cols if c not in ["domain", "sample_uid", "imputation_flag"]},
        **({"imputation_flag": "max"} if "imputation_flag" in meta_cols else {})
    })
)

# Add sequence length check
seq_counts = (
    combined.groupby(["domain", "sample_uid"])
    .size()
    .reset_index(name="n_time_steps")
)

sample_metadata = sample_metadata.merge(seq_counts, on=["domain", "sample_uid"], how="left")

# -----------------------------
# 9) SAVE OUTPUTS
# -----------------------------
aligned_real.to_csv(OUT_REAL, index=False)
aligned_sim.to_csv(OUT_SIM, index=False)
combined.to_csv(OUT_COMBINED, index=False)
combined_labeled.to_csv(OUT_COMBINED_LABELED, index=False)
sample_metadata.to_csv(OUT_META, index=False)

print("\nSaved files:")
print("1)", OUT_REAL)
print("2)", OUT_SIM)
print("3)", OUT_COMBINED)
print("4)", OUT_COMBINED_LABELED)
print("5)", OUT_META)

# -----------------------------
# 10) FINAL SUMMARY
# -----------------------------
print("\n================ SUMMARY ================")
print("Aligned real shape:", aligned_real.shape)
print("Aligned sim shape :", aligned_sim.shape)
print("Combined shape    :", combined.shape)
print("Labeled combined  :", combined_labeled.shape)
print("Metadata shape    :", sample_metadata.shape)

print("\nReal sample labels:")
print(sample_metadata[sample_metadata["domain"] == "real"]["label"].value_counts(dropna=False))

print("\nSimulated sample labels:")
print(sample_metadata[sample_metadata["domain"] == "sim"]["label"].value_counts(dropna=False))

print("\nReal density:")
print(sample_metadata[sample_metadata["domain"] == "real"]["density_clean"].value_counts(dropna=False))

print("\nSim density:")
print(sample_metadata[sample_metadata["domain"] == "sim"]["density_clean"].value_counts(dropna=False))

print("\nReal sequence length check:")
print(sample_metadata[sample_metadata["domain"] == "real"]["n_time_steps"].describe())

print("\nSim sequence length check:")
print(sample_metadata[sample_metadata["domain"] == "sim"]["n_time_steps"].describe())

print("\nDone.")

Real path: /content/drive/MyDrive/CBRA_STATIC_MODELING/INPUT/real_breast_level_40sensors_imputed_final.csv
Sim path : /content/drive/MyDrive/CBRA_STATIC_MODELING/INPUT/calibrated_passive_generated_timeseries_long.csv


/tmp/ipykernel_7111/3109618198.py:35: DtypeWarning: Columns (55,61,66,67,68,71,72,73,79) have mixed types. Specify dtype option on import or set low_memory=False.
  real_df = pd.read_csv(REAL_PATH)



Loaded:
Real shape: (71066, 163)
Sim shape : (329940, 62)

Real samples found: 140


/tmp/ipykernel_7111/3109618198.py:361: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[m] = 1
/tmp/ipykernel_7111/3109618198.py:361: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[m] = 1
/tmp/ipykernel_7111/3109618198.py:361: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[m] = 1
/tmp/ipykerne

Simulated samples found: 564


/tmp/ipykernel_7111/3109618198.py:503: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[m] = 1
/tmp/ipykernel_7111/3109618198.py:503: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[m] = 1
/tmp/ipykernel_7111/3109618198.py:503: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[m] = 1
/tmp/ipykerne


Aligned shapes before derived channels:
Aligned real: (81900, 104)
Aligned sim : (329940, 104)

Adding derived common channels...


/tmp/ipykernel_7111/3109618198.py:204: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"qd_{idx}"] = (df[c].astype("float32") - qmean).astype("float32")
/tmp/ipykernel_7111/3109618198.py:204: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"qd_{idx}"] = (df[c].astype("float32") - qmean).astype("float32")
/tmp/ipykernel_7111/3109618198.py:204: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once u


Saved files:
1) /content/drive/MyDrive/CBRA_STATIC_MODELING/INPUT/aligned_real_imputed_585_long.csv
2) /content/drive/MyDrive/CBRA_STATIC_MODELING/INPUT/aligned_simulated_passive_585_long.csv
3) /content/drive/MyDrive/CBRA_STATIC_MODELING/INPUT/aligned_combined_real_sim_585_long.csv
4) /content/drive/MyDrive/CBRA_STATIC_MODELING/INPUT/aligned_combined_labeled_only.csv
5) /content/drive/MyDrive/CBRA_STATIC_MODELING/INPUT/aligned_sample_metadata.csv

================ SUMMARY ================
Aligned real shape: (81900, 279)
Aligned sim shape : (329940, 279)
Combined shape    : (411840, 279)
Labeled combined  : (404235, 279)
Metadata shape    : (704, 23)

Real sample labels:
label
0.0    87
1.0    47
NaN     6
Name: count, dtype: int64

Simulated sample labels:
label
1.0    329
0.0    228
NaN      7
Name: count, dtype: int64

Real density:
density_clean
B          60
C          42
unknown    30
D           4
A           4
Name: count, dtype: int64

Sim density:
density_clean
B          4